# Bluestock Mutual Fund Capstone — Advanced Analytics & Portfolio Projections
This notebook documents and executes two advanced quantitative models for mutual fund performance analysis:
1. **Monte Carlo Projection (B3)**: Simulating the 5-year NAV growth of a scheme using **Geometric Brownian Motion (GBM)**.
2. **Markowitz Efficient Frontier Portfolio Optimization (B4)**: Simulating random portfolio weights, calculating portfolio risk-return metrics, and identifying optimal allocations for a selected set of 5 funds.

In [ ]:
import os
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
db_path = '../data/db/bluestock_mf.db'
conn = sqlite3.connect(db_path)
print('Connected to SQLite Database!')

## 1. Monte Carlo NAV Projection (B3)
The model assumes that a mutual fund's Net Asset Value (NAV) follows a continuous-time stochastic process called **Geometric Brownian Motion (GBM)**. 
The stochastic differential equation (SDE) is:
$$dS_t = \mu S_t dt + \sigma S_t dW_t$$

Applying Itô's Lemma, we find the closed-form analytical solution:
$$S_t = S_0 \exp\left( \left(\mu - \frac{1}{2}\sigma^2\right)t + \sigma W_t \right)$$

To simulate this in discrete trading days:
$$S_{t+1} = S_t \exp\left( \left(\mu_{daily} - \frac{1}{2}\sigma^2_{daily}\right) + \sigma_{daily} Z_t \right)$$
where $Z_t \sim N(0, 1)$.

In [ ]:
# Load NAV for SBI Bluechip (AMFI: 119551)
df_nav = pd.read_sql_query('''
    SELECT nav_date, nav_value 
    FROM fact_nav 
    WHERE amfi_code = 119551 
    ORDER BY nav_date
''', conn)
df_nav['nav_date'] = pd.to_datetime(df_nav['nav_date'])
df_nav.set_index('nav_date', inplace=True)

# Handle weekends: reindex and forward fill
df_nav = df_nav.reindex(pd.date_range(df_nav.index.min(), df_nav.index.max(), freq='D')).ffill().bfill()
# Daily returns for business days only
df_trading = df_nav[df_nav.index.dayofweek < 5].copy()
df_trading['log_return'] = np.log(df_trading['nav_value'] / df_trading['nav_value'].shift(1))
log_returns = df_trading['log_return'].dropna()

# Parameters
mu_d = log_returns.mean()
sigma_d = log_returns.std()
T_days = int(5 * 252)
N_paths = 500
S0 = 10000

# Simulation
np.random.seed(42)
drift = mu_d - 0.5 * (sigma_d**2)
shocks = np.random.normal(0, 1, (T_days, N_paths))
increments = drift + sigma_d * shocks
cum_inc = np.vstack([np.zeros((1, N_paths)), np.cumsum(increments, axis=0)])
paths = S0 * np.exp(cum_inc)

# Percentiles
p10 = np.percentile(paths, 10, axis=1)
p50 = np.percentile(paths, 50, axis=1)
p90 = np.percentile(paths, 90, axis=1)

# Plot results
plt.figure(figsize=(12, 6))
plt.plot(p50, label='Median Path (50th percentile)', color='indigo', linewidth=2.5)
plt.fill_between(range(T_days + 1), p10, p90, color='indigo', alpha=0.15, label='10th - 90th Percentile Band')
for i in range(5):
    plt.plot(paths[:, i], linewidth=0.7, linestyle='--', alpha=0.6, label=f'Sample Path {i+1}')
plt.title('Monte Carlo 5-Year NAV Projection (SBI Bluechip)', fontsize=13, fontweight='bold')
plt.xlabel('Trading Days')
plt.ylabel('Portfolio Value (₹)')
plt.legend()
plt.show()

print(f'Starting Value: ₹{S0}')
print(f'Ending Median Projected Value: ₹{p50[-1]:.2f}')
print(f'Ending 90th Percentile Value: ₹{p90[-1]:.2f}')
print(f'Ending 10th Percentile Value: ₹{p10[-1]:.2f}')

## 2. Markowitz Portfolio Optimization (B4)
Modern Portfolio Theory (MPT), developed by Harry Markowitz, uses the mean and covariance of asset returns to identify portfolios that maximize expected return for a given level of risk (Standard Deviation).

**Formulations**:
- **Portfolio Expected Return**:
  $$E(R_p) = w^T R$$
- **Portfolio Volatility**:
  $$\sigma_p = \sqrt{w^T \Sigma w}$$
- **Sharpe Ratio**:
  $$SR_p = \frac{E(R_p) - R_f}{\sigma_p}$$
  
We simulate $N = 5000$ portfolios for 5 selected funds to map the Efficient Frontier.

In [ ]:
# Load daily NAVs for 5 selected funds
df_mult = pd.read_sql_query('''
    SELECT n.nav_date, n.nav_value, f.scheme_name 
    FROM fact_nav n
    JOIN dim_fund f ON n.amfi_code = f.amfi_code
    WHERE n.amfi_code IN (125497, 119551, 120503, 118632, 120841)
    ORDER BY n.nav_date
''', conn)
df_pivot = df_mult.pivot(index='nav_date', columns='scheme_name', values='nav_value')
df_pivot.index = pd.to_datetime(df_pivot.index)

# Handle weekends: reindex and forward fill
df_pivot = df_pivot.reindex(pd.date_range(df_pivot.index.min(), df_pivot.index.max(), freq='D')).ffill().bfill()
# Calculate daily returns for business days only
df_ret = df_pivot[df_pivot.index.dayofweek < 5].pct_change().dropna()

# Annualized returns and covariance
ann_returns = df_ret.mean() * 252
ann_cov = df_ret.cov() * 252

# Portfolio Simulation
num_ports = 5000
port_returns = []
port_vols = []
port_weights = []
rf = 0.065

np.random.seed(101)
for i in range(num_ports):
    w = np.random.random(5)
    w /= np.sum(w)
    p_ret = np.sum(w * ann_returns)
    p_vol = np.sqrt(np.dot(w.T, np.dot(ann_cov, w)))
    
    port_returns.append(p_ret)
    port_vols.append(p_vol)
    port_weights.append(w)

port_returns = np.array(port_returns)
port_vols = np.array(port_vols)
sharpe_ratios = (port_returns - rf) / port_vols

# Find Max Sharpe & Min Volatility
max_sh_idx = sharpe_ratios.argmax()
min_vol_idx = port_vols.argmin()

# Plotting
plt.figure(figsize=(10, 6))
plt.scatter(port_vols * 100, port_returns * 100, c=sharpe_ratios, cmap='viridis', marker='o', s=10, alpha=0.8)
plt.colorbar(label='Sharpe Ratio')
plt.scatter(port_vols[max_sh_idx]*100, port_returns[max_sh_idx]*100, marker='*', color='red', s=200, label='Max Sharpe Ratio')
plt.scatter(port_vols[min_vol_idx]*100, port_returns[min_vol_idx]*100, marker='*', color='green', s=200, label='Minimum Volatility')
plt.title('Markowitz Efficient Frontier Portfolio Optimization', fontsize=13, fontweight='bold')
plt.xlabel('Annualized Volatility (Standard Deviation %)')
plt.ylabel('Expected Annualized Return (%)')
plt.legend()
plt.show()

print('Max Sharpe Portfolio Allocation:')
for fund, weight in zip(df_pivot.columns, port_weights[max_sh_idx]):
    print(f'  {fund}: {weight*100:.2f}%')
print(f'Expected Return: {port_returns[max_sh_idx]*100:.2f}%, Volatility: {port_vols[max_sh_idx]*100:.2f}%, Sharpe: {sharpe_ratios[max_sh_idx]:.2f}')

conn.close()